Before running the notebook run the command `mlflow server --host 127.0.0.1 --port 5000`

In [1]:
from pathlib import Path

import mlflow
import pandas as pd
from hydra import compose, initialize
from pycaret import classification


# Load config

In [2]:
with initialize(version_base=None, config_path="../conf"):
    config = compose(config_name="health_prediction_config")

# Load data

In [3]:
health_df = pd.read_csv(Path("..") / config.silver_dataset_path)
health_df

,Age,Gender,Cholesterol,Glucose,Smoking,Alcohol Consumption,Exercise,BMI,Family History,Heart Disease,...,Stroke,Kidney Disease,Cancer,Alzheimer's Disease,COPD,Liver Disease,Parkinson's Disease,Tuberculosis,Blood Pressure_Low,Blood Pressure_Normal
0,69,0,1,1,1,0,0,35.671099,0,1,...,0,0,1,0,0,0,0,0,0,0
1,32,0,1,0,1,0,1,38.554188,1,0,...,0,0,0,0,0,1,0,0,1,0
2,89,1,1,0,0,0,1,18.932964,1,1,...,0,0,0,0,0,0,0,0,0,1
3,78,0,1,1,0,0,1,21.806350,1,0,...,1,0,1,0,0,1,0,0,0,0
4,38,0,0,0,1,1,1,37.552683,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,27,1,1,0,0,0,0,31.960176,1,1,...,0,0,0,0,0,0,0,0,1,0
996,51,1,1,0,0,1,1,20.118492,1,0,...,0,0,0,0,0,0,0,0,0,0
997,72,1,1,0,1,0,0,20.916536,1,0,...,0,0,0,0,0,0,0,0,0,1
998,49,0,1,1,1,0,1,19.560143,1,0,...,0,0,0,1,0,0,0,0,0,1


# Modelling

In [4]:
# Connect to Mlflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")


# Initialize PyCaret Setup and train models
target_cols = ["Heart Disease", "Diabetes", "Stroke", "Kidney Disease", "Cancer", "Alzheimer's Disease", "COPD", "Liver Disease", "Parkinson's Disease", "Tuberculosis"]

reg_setups = {
    "with": {},
    "without": {},
}

best_models = {
    "with": {},
    "without": {},
}

comp_results = {
    "with": {},
    "without": {},
}


for col in target_cols:
    experiment_name = "health_" + col.lower().replace(" ", "_").replace("'", " ") + "_with"

    reg_setups["with"][col] = classification.setup(
        data=health_df,
        target=col,
        train_size=config.train_size,
        fix_imbalance=True,
        transformation=True,
        normalize=True,
        session_id=config.random_state,
        log_experiment=True,
        experiment_name=experiment_name,
        memory=False,
    )
    best_models["with"][col] = classification.compare_models(sort="F1")
    comp_results["with"][col] = classification.pull().copy()

    reg_setups["without"][col] = classification.setup(
        data=health_df,
        target=col,
        train_size=config.train_size,
        ignore_features=["Age", "BMI"],
        fix_imbalance=True,
        transformation=True,
        normalize=True,
        session_id=config.random_state,
        log_experiment=True,
        experiment_name=experiment_name + "out",
        memory=False,
    )
    best_models["without"][col] = classification.compare_models(sort="F1")
    comp_results["without"][col] = classification.pull().copy()


,Description,Value
0,Session id,67
1,Target,Heart Disease
2,Target type,Binary
3,Original data shape,"(1000, 21)"
4,Transformed data shape,"(1396, 21)"
5,Transformed train set shape,"(1196, 21)"
6,Transformed test set shape,"(200, 21)"
7,Numeric features,20
8,Preprocess,True
9,Imputation type,simple


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
knn,K Neighbors Classifier,0.5262,0.4939,0.4462,0.2497,0.3167,-0.0025,-0.0005,1.3720
lr,Logistic Regression,0.5725,0.5196,0.3712,0.2627,0.3056,0.0125,0.0125,0.7050
ridge,Ridge Classifier,0.5775,0.5194,0.3662,0.2632,0.3046,0.0147,0.0149,0.1790
lda,Linear Discriminant Analysis,0.5775,0.5194,0.3662,0.2632,0.3046,0.0147,0.0149,0.1780
svm,SVM - Linear Kernel,0.5812,0.4992,0.3614,0.2608,0.2985,0.0140,0.0154,0.1910
nb,Naive Bayes,0.5925,0.5197,0.3121,0.2490,0.2748,-0.0017,-0.0017,0.7590
dt,Decision Tree Classifier,0.5963,0.4942,0.2879,0.2424,0.2619,-0.0125,-0.0122,0.1470
qda,Quadratic Discriminant Analysis,0.6075,0.5184,0.2626,0.2408,0.2482,-0.0133,-0.0139,0.1080
xgboost,Extreme Gradient Boosting,0.6838,0.5005,0.1686,0.2869,0.2106,0.0310,0.0327,0.5400
et,Extra Trees Classifier,0.6788,0.4666,0.1240,0.2394,0.1606,-0.0111,-0.0120,0.5480


2026/08/21 14:31:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 14:31:17 INFO mlflow.tracking._tracking_service.client: 🏃 View run K Neighbors Classifier at: http://127.0.0.1:5000/#/experiments/296219890235609537/runs/ed45d3bd361241d4aba6e239f8e3b85e.
2026/08/21 14:31:17 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/296219890235609537.
2026/08/21 14:31:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 14:31:18 INFO mlflow.tracking._tracking_service.client: 🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/296219890235609537/runs/3555f4f4de534a1fad682d64975d504a.
2026/08/21 14:31:18 INFO mlflow.tracking._tracking_service.client: 🧪

,Description,Value
0,Session id,67
1,Target,Heart Disease
2,Target type,Binary
3,Original data shape,"(1000, 21)"
4,Transformed data shape,"(1396, 19)"
5,Transformed train set shape,"(1196, 19)"
6,Transformed test set shape,"(200, 19)"
7,Ignore features,2
8,Numeric features,18
9,Preprocess,True


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.5412,0.5369,0.5007,0.2760,0.3545,0.0440,0.0491,0.0170
lda,Linear Discriminant Analysis,0.5412,0.5368,0.5007,0.2760,0.3545,0.0440,0.0491,0.0200
lr,Logistic Regression,0.5400,0.5369,0.5010,0.2748,0.3535,0.0422,0.0478,0.0330
nb,Naive Bayes,0.4850,0.5148,0.5298,0.2516,0.3404,-0.0010,-0.0002,0.0390
qda,Quadratic Discriminant Analysis,0.5575,0.5051,0.4410,0.2736,0.3349,0.0341,0.0353,0.0380
svm,SVM - Linear Kernel,0.5438,0.4974,0.3917,0.2485,0.3020,-0.0088,-0.0109,0.0260
dt,Decision Tree Classifier,0.6300,0.5066,0.3029,0.2786,0.2888,0.0403,0.0405,0.0250
knn,K Neighbors Classifier,0.5000,0.4431,0.3764,0.2160,0.2733,-0.0664,-0.0723,0.0200
ada,Ada Boost Classifier,0.6862,0.5408,0.1981,0.3181,0.2360,0.0556,0.0607,0.0430
xgboost,Extreme Gradient Boosting,0.6550,0.4783,0.1736,0.2312,0.1974,-0.0128,-0.0149,0.0360


...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
svm,SVM - Linear Kernel,0.7337,0.5017,0.3500,0.0457,0.0800,0.0202,0.0395,0.0550
nb,Naive Bayes,0.7050,0.5890,0.3167,0.0451,0.0772,0.0186,0.0116,0.1160
ridge,Ridge Classifier,0.7325,0.5001,0.3167,0.0443,0.0771,0.0170,0.0286,0.0730
lda,Linear Discriminant Analysis,0.7325,0.5001,0.3167,0.0443,0.0771,0.0170,0.0286,0.0690
lr,Logistic Regression,0.7312,0.4977,0.3167,0.0428,0.0750,0.0147,0.0266,0.2320
knn,K Neighbors Classifier,0.7312,0.4809,0.2833,0.0366,0.0645,0.0038,0.0112,1.2130
qda,Quadratic Discriminant Analysis,0.9175,0.5833,0.0333,0.0250,0.0286,0.0030,-0.0030,0.0320
dt,Decision Tree Classifier,0.9275,0.4806,0.0000,0.0000,0.0000,-0.0341,-0.0359,0.0360
rf,Random Forest Classifier,0.9625,0.5038,0.0000,0.0000,0.0000,-0.0038,-0.0044,0.1680
ada,Ada Boost Classifier,0.9512,0.5763,0.0000,0.0000,0.0000,-0.0182,-0.0200,0.1060


2026/08/21 14:55:04 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 14:55:04 INFO mlflow.tracking._tracking_service.client: 🏃 View run SVM - Linear Kernel at: http://127.0.0.1:5000/#/experiments/668694462194489806/runs/ccd7e059a5af43ba906d4bb0433e26fb.
2026/08/21 14:55:04 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/668694462194489806.
2026/08/21 14:55:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 14:55:05 INFO mlflow.tracking._tracking_service.client: 🏃 View run Naive Bayes at: http://127.0.0.1:5000/#/experiments/668694462194489806/runs/ad21fd7b83d9428994624bd8cbbd20e7.
2026/08/21 14:55:05 INFO mlflow.tracking._tracking_service.client: 🧪 View exper

,Description,Value
0,Session id,67
1,Target,Tuberculosis
2,Target type,Binary
3,Original data shape,"(1000, 21)"
4,Transformed data shape,"(1744, 19)"
5,Transformed train set shape,"(1544, 19)"
6,Transformed test set shape,"(200, 19)"
7,Ignore features,2
8,Numeric features,18
9,Preprocess,True


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
svm,SVM - Linear Kernel,0.6712,0.4631,0.2833,0.0339,0.0598,-0.0025,-0.0097,0.0470
nb,Naive Bayes,0.6038,0.4939,0.3333,0.0332,0.0596,-0.0038,-0.0277,0.0510
ridge,Ridge Classifier,0.6825,0.4613,0.2833,0.0325,0.0581,-0.0046,-0.0075,0.0480
lda,Linear Discriminant Analysis,0.6825,0.4613,0.2833,0.0325,0.0581,-0.0046,-0.0075,0.0510
lr,Logistic Regression,0.6788,0.4639,0.2833,0.0323,0.0578,-0.0050,-0.0089,0.0480
knn,K Neighbors Classifier,0.7450,0.4329,0.2167,0.0321,0.0554,-0.0051,-0.0080,0.0590
dt,Decision Tree Classifier,0.8988,0.4774,0.0333,0.0143,0.0200,-0.0239,-0.0266,0.0640
qda,Quadratic Discriminant Analysis,0.8475,0.4805,0.0500,0.0014,0.0027,-0.0294,-0.0483,0.0650
rf,Random Forest Classifier,0.9588,0.4779,0.0000,0.0000,0.0000,-0.0093,-0.0107,0.2570
ada,Ada Boost Classifier,0.9625,0.4079,0.0000,0.0000,0.0000,-0.0031,-0.0032,0.1180


2026/08/21 14:56:41 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 14:56:42 INFO mlflow.tracking._tracking_service.client: 🏃 View run SVM - Linear Kernel at: http://127.0.0.1:5000/#/experiments/479745575655938716/runs/3744a334b1a34b5eb78d982457dbfd09.
2026/08/21 14:56:42 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/479745575655938716.
2026/08/21 14:56:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/08/21 14:56:43 INFO mlflow.tracking._tracking_service.client: 🏃 View run Naive Bayes at: http://127.0.0.1:5000/#/experiments/479745575655938716/runs/03676fdaa0dc44a596a9f89e93d3c730.
2026/08/21 14:56:43 INFO mlflow.tracking._tracking_service.client: 🧪 View exper

## Compare Models

In [6]:
performance_comparison = pd.DataFrame(
    {
        "With Age and BMI": {disease: results["F1"].max() for disease, results in comp_results["with"].items()},
        "Without Age and BMI": {disease: results["F1"].max() for disease, results in comp_results["without"].items()},
    }
)

performance_comparison["Better Setup"] = performance_comparison.idxmax(axis=1)
performance_comparison


,With Age and BMI,Without Age and BMI,Better Setup
Heart Disease,0.3167,0.3545,Without Age and BMI
Diabetes,0.2742,0.2821,Without Age and BMI
Stroke,0.2127,0.1996,With Age and BMI
Kidney Disease,0.2131,0.2341,Without Age and BMI
Cancer,0.1886,0.1617,With Age and BMI
Alzheimer's Disease,0.2171,0.2415,Without Age and BMI
COPD,0.2188,0.2117,With Age and BMI
Liver Disease,0.2465,0.2266,With Age and BMI
Parkinson's Disease,0.1277,0.1219,With Age and BMI
Tuberculosis,0.0800,0.0598,With Age and BMI
